# 18 · Files and Streams

*Beyond the contest* · 6 exercises

Read and write real files: numbers and words line by line, reports and CSV records, and streams that handle a file of any size without loading it whole.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ch-18) · [Answer key](https://bloomcode.github.io/cp/python/answer-key.html#ch-18)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.
import csv
import re
import tempfile
from pathlib import Path

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    class _Raises:
        def __init__(self, kind):
            self.kind = kind

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            return issubclass(kind, self.kind)

    class pytest:  # noqa: N801
        raises = _Raises

### 18.1 · Sum the numbers in a file  `Core`

**Task.** Write `sum_file(path)` that returns the sum of the integers in a text file, one per line, skipping blank lines. Open it with `with open(path) as f` and iterate over `f`, which yields one line at a time.

**Example.** a file holding `3\n\n-1\n10\n` -> `12`

**Big O.** O(size of the file) time, O(1) extra space when read line by line.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-18-1) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-18-1)

In [ ]:
# 18.1 Sum the numbers in a file
def sum_file(path):
    raise NotImplementedError

In [ ]:
def test_ex_18_1():
    with tempfile.TemporaryDirectory() as name:
        tmp = Path(name)
        (tmp / "n.txt").write_text("3\n\n-1\n10\n")
        (tmp / "empty.txt").write_text("")
        assert sum_file(tmp / "n.txt") == 12 and sum_file(tmp / "empty.txt") == 0


test_ex_18_1()
print('18.1 passed')

### 18.2 · Most common words in a file  `Core`

**Task.** A word is a run of letters, compared in lower case. Write `top_words(path, k)` that returns the `k` most common words in a file as `(word, count)` pairs, most frequent first and ties in alphabetical order. Count with a dictionary (or `collections.Counter`), then sort by `(-count, word)`.

**Example.** `The cat and the hat.\nThe end, and THE start!\n`, `2` -> `[('the', 4), ('and', 2)]`

**Big O.** O(size + W log W) for W distinct words: one pass to count, one sort.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-18-2) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-18-2)

In [ ]:
# 18.2 Most common words in a file
def top_words(path, k):
    raise NotImplementedError

In [ ]:
def test_ex_18_2():
    with tempfile.TemporaryDirectory() as name:
        tmp = Path(name)
        (tmp / "w.txt").write_text("The cat and the hat.\nThe end, and THE start!\n")
        assert top_words(tmp / "w.txt", 2) == [("the", 4), ("and", 2)]
        assert top_words(tmp / "w.txt", 4) == [("the", 4), ("and", 2), ("cat", 1), ("end", 1)] and len(top_words(tmp / "w.txt", 99)) == 6


test_ex_18_2()
print('18.2 passed')

### 18.3 · Write a report  `Core`

**Task.** Write `write_report(path, scores)` that writes one line `name,score` per entry of the dictionary `scores`, highest score first and ties by name, replacing any existing file, and returns how many lines it wrote. Open with mode `"w"` and write each line with `\n`.

**Example.** `{'ana': 7, 'bo': 9, 'cy': 7}` -> writes `bo,9\nana,7\ncy,7\n` and returns `3`

**Big O.** O(N log N) for the sort, O(N) to write.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-18-3) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-18-3)

In [ ]:
# 18.3 Write a report
def write_report(path, scores):
    raise NotImplementedError

In [ ]:
def test_ex_18_3():
    with tempfile.TemporaryDirectory() as name:
        tmp = Path(name)
        path = tmp / "report.csv"
        path.write_text("old contents\n")
        assert write_report(path, {"ana": 7, "bo": 9, "cy": 7}) == 3 and path.read_text() == "bo,9\nana,7\ncy,7\n"
        assert write_report(path, {}) == 0 and path.read_text() == ""


test_ex_18_3()
print('18.3 passed')

### 18.4 · Stream a large file  `Practice`

**Task.** Write `longest_line(path)` that returns the length of the longest line in a file (not counting the newline), holding only one line in memory at a time: iterate over the open file, never call `read()` or `readlines()`.

**Example.** a file with lines `ab`, `abcd`, `` -> `4`

**Big O.** O(size) time and O(longest line) memory: only one line is ever held, however big the file is.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-18-4) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-18-4)

In [ ]:
# 18.4 Stream a large file
def longest_line(path):
    raise NotImplementedError

In [ ]:
def test_ex_18_4():
    with tempfile.TemporaryDirectory() as name:
        tmp = Path(name)
        (tmp / "a.txt").write_text("ab\nabcd\n\n")
        with open(tmp / "big.txt", "w") as f:
            for i in range(100_000):
                f.write("x" * (i % 97) + "\n")
            f.write("y" * 500)
        assert longest_line(tmp / "a.txt") == 4 and longest_line(tmp / "big.txt") == 500


test_ex_18_4()
print('18.4 passed')

### 18.5 · Copy matching lines to another file  `Practice`

**Task.** Write `copy_matching(source, target, needle)` that writes every line of `source` containing `needle` to `target` (in order, newlines kept) and returns how many it wrote. Open both files in one `with` statement and stream from one to the other.

**Example.** lines `error: disk`, `ok`, `error: net` with needle `error` -> writes the two error lines, returns `2`

**Big O.** O(size of the input) time, O(longest line) memory: both files are streamed.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-18-5) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-18-5)

In [ ]:
# 18.5 Copy matching lines to another file
def copy_matching(source, target, needle):
    raise NotImplementedError

In [ ]:
def test_ex_18_5():
    with tempfile.TemporaryDirectory() as name:
        tmp = Path(name)
        (tmp / "log.txt").write_text("error: disk\nok\nerror: net")
        assert copy_matching(tmp / "log.txt", tmp / "out.txt", "error") == 2 and (tmp / "out.txt").read_text() == "error: disk\nerror: net"
        assert copy_matching(tmp / "log.txt", tmp / "none.txt", "panic") == 0 and (tmp / "none.txt").read_text() == ""


test_ex_18_5()
print('18.5 passed')

### 18.6 · Averages from a CSV file  `Practice`

**Task.** A CSV file has the header `team,points` and one row per game. Write `average_by_team(path)` that returns a dictionary from team to its average points (a float). Read the rows with `csv.DictReader` (open the file with `newline=""`) and keep a total and a count per team.

**Example.** rows `red,10`, `blue,4`, `red,7` -> `{'red': 8.5, 'blue': 4.0}`

**Big O.** O(size of the file) time, O(T) space for T teams: keep a running total and count per team.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-18-6) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-18-6)

In [ ]:
# 18.6 Averages from a CSV file
def average_by_team(path):
    raise NotImplementedError

In [ ]:
def test_ex_18_6():
    with tempfile.TemporaryDirectory() as name:
        tmp = Path(name)
        (tmp / "games.csv").write_text("team,points\nred,10\nblue,4\nred,7\n")
        (tmp / "none.csv").write_text("team,points\n")
        assert average_by_team(tmp / "games.csv") == {"red": 8.5, "blue": 4.0} and average_by_team(tmp / "none.csv") == {}
        (tmp / "quoted.csv").write_text('team,points\n"red, the best",3\n')
        assert average_by_team(tmp / "quoted.csv") == {"red, the best": 3.0}


test_ex_18_6()
print('18.6 passed')